In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import sys
import algos.net2 as nn2
import matplotlib.pyplot as plt



def imshow(img, cmap='gray'):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)), cmap=cmap)
    plt.show()



transform = transforms.Compose(
    [
        transforms.Grayscale(),  # Convert the image to a PyTorch tensor
        transforms.ToPILImage(),
        transforms.Resize((28, 28)),  # Resize the image to 28x28
        transforms.Normalize((0.5,), (0.5,)),  # Normalize images
    ]
)

# Download and load the training and test datasets
train_dataset = datasets.MNIST(
    root="./data", train=True, transform=transform, download=True
)
test_dataset = datasets.MNIST(
    root="./data", train=False, transform=transform, download=True
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
imshow(train_dataset[0][0])  # Display the first training image

TypeError: pic should be Tensor or ndarray. Got <class 'PIL.Image.Image'>.

In [ ]:
def display_model_predictions(model, test_loader, num_images=5, cmap='gray', classes_dict=None):
    images, labels = next(iter(test_loader))
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    if classes_dict is not None:
        labels = [classes_dict[l.item()] for l in labels]
        predicted = [classes_dict[p.item()] for p in predicted]
    
    rows = int(np.ceil(num_images / 5))
    columns = 5
    _, axes = plt.subplots(rows, columns, figsize=(12, 6))
    
    for i in range(rows * columns):
        if i < num_images:
            current_row = i // 5
            image = images[i] / 2 + 0.5  # Unnormalize
            if rows > 1:
                axes[current_row, i % 5].imshow(image.permute(1, 2, 0).numpy(), cmap=cmap)
                axes[current_row, i % 5].set_title(f"Label: {labels[i]},\nPredicted: {predicted[i]}")
                axes[current_row, i % 5].axis("off")
            else:
                axes[i].imshow(image.permute(1, 2, 0).numpy(), cmap=cmap)
                axes[i].set_title(f"Label: {labels[i]},\nPredicted: {predicted[i]}")
                axes[i].axis("off")
        else:
            if rows > 1:
                axes[current_row, i % 5].axis("off")
            else:
                axes[i].axis("off")
    plt.tight_layout()
    plt.show()
    
    
network = nn2.NeuralNet(
    layers=[
        nn2.Flatten(),
        nn2.FullyConnected(28 * 28, 128),
        nn2.ReLU(),
        nn2.FullyConnected(128, 64),
        nn2.ReLU(),
        nn2.Dropout(0.2),
        nn2.FullyConnected(64, 10),
        nn2.Softmax(),
    ]
)